# Phase 2C.5 - C3 Winner Reserve Final (Colab)

Run exactly one approved C3 hierarchical finalist on the frozen 587-question generation reserve from 100 articles. The winner must be selected from the shared 281-question development comparison before this notebook is enabled. The earlier 284-question Phase 2 held-out set is already open and is not used here. Reserve results measure winner generalization and must not revise the winner. A T4 GPU is required for fresh BGE-M3 retrieval and BGE-large reranking.

Before execution: place both C3 development-result ZIPs at the configured Drive paths, set `LOCKED_WINNER_DEPTH` to `3` or `5`, complete all `WINNER_REVIEW` fields, then set `EXECUTE_FINAL=True`. Configure Colab secrets `HF_TOKEN`, `GEMINI_API_KEY_1`, and `FIREWORKS_API_KEY`. Do not run the losing configuration on the reserve.

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='19a2da4429da2989973777013be77e9b7378a7af'
ARM_ID='c3_hierarchical'; PROMPT_ID='p2'
LOCKED_WINNER_DEPTH=None          # Set to 3 or 5 only after reviewing both 281-question finalist results.
EXECUTE_FINAL=False               # Keep False until the winner decision below is complete.
FINALIST_D3_RESULTS_PATH='/content/drive/MyDrive/newsqa_phase2c/results/phase2c_finalist_c3_d3_development_results.zip'
FINALIST_D5_RESULTS_PATH='/content/drive/MyDrive/newsqa_phase2c/results/phase2c_finalist_c3_d5_development_results.zip'
WINNER_REVIEW={'reviewer_id':'','approved_at':'','notes':''}  # Notes must document comparison against C0-P2-D3/D5 and the other C3 finalist.

PHASE2C_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2c-indexes-v1'
PHASE2C_REVISION='09421ba33e75f2cef01dd01451ce4523214589ca'
PHASE2C_FILENAME='phase2c_chunking_indexes_v1.zip'
PHASE2C_SHA256='96030993b38de4d62828ce3d9945ae8fc73647c7c8fd0155d6eb223d0e6a0c90'
PREP_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
PREP_REVISION='771fee6e19e3e836c9ba0b15874fe0bd5199b2a5'
PREP_FILENAME='phase2b/preparation-refined-prompts-v1/phase2b_preparation_bundle.zip'
PREP_SHA256='0b2ab8388d6ab679970519bc35ba574936a063b32406ce5cee4b6ec5c166073c'
GENERATOR_MODEL='gemini-3.1-flash-lite'; GENERATOR_REASONING='minimal'; GENERATOR_MAX_TOKENS=512; GENERATOR_INTERVAL=4.2
JUDGE_MODEL='accounts/fireworks/models/glm-5p3-flash'; JUDGE_REASONING='low'; JUDGE_MAX_TOKENS=2048
GEMINI_SECRET_NAME='GEMINI_API_KEY_1'; SEED=42; TOP_K=20; RERANK_TOP_N=5
GENERATOR_INPUT_USD_PER_MILLION=0.25; GENERATOR_OUTPUT_USD_PER_MILLION=1.50; JUDGE_INPUT_USD_PER_MILLION=0.15; JUDGE_OUTPUT_USD_PER_MILLION=0.50
ROOT=Path('/content'); PROJECT_ROOT=ROOT/'Text-Mining---NewsQA-RAG'; WORK=ROOT/'phase2c_c3_reserve_final'
PERSIST=ROOT/'drive/MyDrive/newsqa_phase2c/reserve_final'
DATA=WORK/'data'; RUNTIME=WORK/'runtime'; CHILD_TRACE=PERSIST/'child_trace'; RUN_DIR=PERSIST/'run'; RESULTS=PERSIST/'results'; LOGS=PERSIST/'logs'

## 1. Setup and immutable inputs

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
from google.colab import drive,userdata
drive.mount('/content/drive')
for path in [DATA,RUNTIME,CHILD_TRACE,RUN_DIR,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
assert ARM_ID=='c3_hierarchical' and PROMPT_ID=='p2'
assert not REPO_COMMIT.startswith('SET_TO_'),'Pin REPO_COMMIT after committing these notebooks'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import pandas as pd,yaml
from huggingface_hub import hf_hub_download
def secret(name):
    try: return userdata.get(name) or ''
    except Exception: return ''
HF_TOKEN=secret('HF_TOKEN'); GENERATOR_KEY=secret(GEMINI_SECRET_NAME); JUDGE_KEY=secret('FIREWORKS_API_KEY')
if EXECUTE_FINAL: assert GENERATOR_KEY and JUDGE_KEY,f'Configure {GEMINI_SECRET_NAME} and FIREWORKS_API_KEY'
assert HF_TOKEN,'HF_TOKEN is required for the private preparation repository'
os.environ.update({'PYTHONPATH':str(PROJECT_ROOT/'common'),'PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false','TOKENIZERS_PARALLELISM':'false'})
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def rows(path):
    with Path(path).open(encoding='utf-8') as f: return [json.loads(line) for line in f if line.strip()]
def stable_hash(value): return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def extract(archive,target,marker,prefix=None):
    target.mkdir(parents=True,exist_ok=True)
    if not (target/marker).exists():
        with zipfile.ZipFile(archive) as z:
            members=[item for item in z.infolist() if prefix is None or item.filename.startswith(prefix)]
            z.extractall(target,members)
    assert (target/marker).exists()
p2c_zip=Path(hf_hub_download(repo_id=PHASE2C_REPO_ID,repo_type='dataset',revision=PHASE2C_REVISION,filename=PHASE2C_FILENAME)); assert sha(p2c_zip)==PHASE2C_SHA256
prep_zip=Path(hf_hub_download(repo_id=PREP_REPO_ID,repo_type='dataset',revision=PREP_REVISION,filename=PREP_FILENAME,token=HF_TOKEN)); assert sha(prep_zip)==PREP_SHA256
P2C=DATA/'phase2c'; PREP=DATA/'preparation'; folder='hierarchical'
extract(p2c_zip,P2C,f'{folder}/artifact_manifest.json',f'{folder}/'); extract(prep_zip,PREP,'question_ids/heldout_reserve.json')
reserve_ids=json.loads((PREP/'question_ids/heldout_reserve.json').read_text()); assert len(reserve_ids)==587 and len(set(reserve_ids))==587
development_ids=json.loads((PREP/'question_ids/development.json').read_text()); opened_heldout_ids=json.loads((PREP/'question_ids/heldout.json').read_text()); assert len(development_ids)==281 and len(opened_heldout_ids)==284 and set(reserve_ids).isdisjoint(development_ids) and set(reserve_ids).isdisjoint(opened_heldout_ids)
IDS=RUNTIME/'reserve_ids.json'; IDS.write_text(json.dumps(reserve_ids)); JUDGE_IDS=IDS
DEVELOPMENT_IDS=RUNTIME/'development_ids.json'; DEVELOPMENT_IDS.write_text(json.dumps(development_ids))
def zip_json(path,name):
    with zipfile.ZipFile(path) as archive:
        matches=[item for item in archive.namelist() if item==name or item.endswith('/'+name)]
        assert len(matches)==1,f'Expected one {name} in {path}'
        return json.loads(archive.read(matches[0]))
finalist_paths={3:Path(FINALIST_D3_RESULTS_PATH),5:Path(FINALIST_D5_RESULTS_PATH)}
finalist_records={}
for depth,path in finalist_paths.items():
    assert path.exists(),f'Missing C3-D{depth} finalist result: {path}'
    summary=zip_json(path,'summary.json'); lock=zip_json(path,'finalist_lock.json')
    assert summary['arm']==ARM_ID and summary['prompt_id']==PROMPT_ID and summary['context_depth']==depth
    assert summary['run_mode']=='development' and summary['questions']==281 and summary['judge_questions']==281
    assert summary['coverage']['successful']==281 and summary['ragas']['n_samples']==281
    assert lock['status']=='locked_finalist' and lock['context_depth']==depth and lock['question_ids_sha256']==sha(DEVELOPMENT_IDS)
    finalist_records[depth]={'depth':depth,'path':str(path),'sha256':sha(path),'summary':summary}
assert LOCKED_WINNER_DEPTH in {3,5},'Set LOCKED_WINNER_DEPTH only after both finalist results are reviewed'
assert all(WINNER_REVIEW.get(key) for key in ('reviewer_id','approved_at','notes')),'Complete WINNER_REVIEW before reserve access'
winner_decision={'schema_version':1,'status':'approved','selected_on_partition':'development','selection_scope':['c0_p2_d3','c0_p2_d5','c3_p2_d3','c3_p2_d5'],'reserve_outputs_accessed_for_selection':False,'opened_phase2_heldout_used_for_selection':False,'winner':{'arm':ARM_ID,'prompt_id':PROMPT_ID,'context_depth':LOCKED_WINNER_DEPTH},'validated_c3_finalists':[{k:v for k,v in finalist_records[d].items() if k!='summary'} for d in (3,5)],'review':WINNER_REVIEW}
decision_path=RESULTS/'phase2c_winner_decision.json'
if decision_path.exists(): assert json.loads(decision_path.read_text())==winner_decision,'Existing reserve decision differs; use a new Drive folder'
else: decision_path.write_text(json.dumps(winner_decision,indent=2,sort_keys=True)+'\n')
print('Locked winner:',winner_decision['winner'],'| reserve questions:',len(reserve_ids))

## 2. Bind C3 artifacts and build child/parent evaluation profiles

In [ ]:
ARM_ROOT=P2C/folder; chunks=ARM_ROOT/'chunks.jsonl'; parents=ARM_ROOT/'parents.jsonl'; mapping=ARM_ROOT/'child_parent_map.jsonl'; testset_child=ARM_ROOT/'testset_resolved.jsonl'; sparse_index=ARM_ROOT/'bge_m3_sparse.pkl'
artifact_manifest=json.loads((ARM_ROOT/'artifact_manifest.json').read_text()); assert artifact_manifest['arm_id']==ARM_ID and artifact_manifest['statistics']['questions']==1152
for name,record in artifact_manifest['artifacts'].items(): path=ARM_ROOT/name; assert path.exists() and path.stat().st_size==record['bytes'] and sha(path)==record['sha256'],name
parent_by_id={r['id']:r for r in rows(parents)}; child_parent={r['child_id']:r['parent_id'] for r in rows(mapping)}
test_rows=rows(testset_child); test_ids={r['question_id'] for r in test_rows}; assert len(test_rows)==1152 and set(reserve_ids)<=test_ids
transformed=[]
for row in test_rows:
    row=dict(row); row['relevant_chunk_ids']=list(dict.fromkeys(child_parent[cid] for cid in row['relevant_chunk_ids'])); transformed.append(row)
testset_parent=RUNTIME/'testset_resolved_parent.jsonl'
with testset_parent.open('w',encoding='utf-8') as out:
    for row in transformed: out.write(json.dumps(row)+'\n')
config=yaml.safe_load((ARM_ROOT/'config.yaml').read_text()); config['llm'].update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS,'reasoning_effort':GENERATOR_REASONING}); config['retrieval']['sparse']['device']='cuda:0'; config['retrieval']['reranker'].update({'enabled':True,'type':'cross-encoder','model':'BAAI/bge-reranker-large','top_n':RERANK_TOP_N,'batch_size':8,'device':'cuda:0'}); config_path=RUNTIME/'config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False))
def profile_for(testset,path):
    variant=json.loads((ARM_ROOT/'variant.json').read_text()); variant['pipeline'].update({'config_path':str(config_path),'config_sha256':stable_hash(config)}); variant['artifacts']['chunks']={'path':str(chunks),'sha256':sha(chunks)}; variant['artifacts']['bm25']={'path':str(sparse_index),'sha256':sha(sparse_index)}; variant['artifacts']['testset_resolved']={'path':str(testset),'sha256':sha(testset)}; path.write_text(json.dumps(variant,indent=2,sort_keys=True)+'\n'); return path
child_profile=profile_for(testset_child,RUNTIME/'child_variant.json'); parent_profile=profile_for(testset_parent,RUNTIME/'parent_variant.json')
prompt=PREP/'prompts/p2.txt'; assert prompt.exists(); print('C3 artifact and reserve mappings verified')

## 3. One-time resumable reserve execution

In [ ]:
def run(command,label,extra_env=None):
    log=LOGS/f'{label}.log'; print('$',' '.join(map(str,command)),flush=True); env={**os.environ,**(extra_env or {})}
    with log.open('a',encoding='utf-8') as out:
        p=subprocess.Popen(list(map(str,command)),cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); out.write(line); out.flush()
        code=p.wait()
    if code: raise subprocess.CalledProcessError(code,command)
assert EXECUTE_FINAL,'Set EXECUTE_FINAL=True only after the C3 winner decision is frozen'
import torch
assert torch.cuda.is_available(),'Enable a T4 GPU for C3 retrieval and reranking'
access_path=RESULTS/'reserve_access.json'; access={'schema_version':1,'status':'started','winner_decision_sha256':sha(decision_path),'reserve_ids_sha256':sha(IDS),'questions':587}
if access_path.exists():
    previous=json.loads(access_path.read_text()); assert all(previous.get(k)==v for k,v in access.items() if k!='status'),'Reserve access contract changed'; access['started_at']=previous['started_at']
else: access['started_at']=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()); access_path.write_text(json.dumps(access,indent=2,sort_keys=True)+'\n')
retrieve=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset_child,'--variant-manifest',child_profile,'--config',config_path,'--run-dir',CHILD_TRACE,'--question-ids-file',IDS,'--chunks-path',chunks,'--bm25-path',sparse_index,'--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--rerank-candidate-n',TOP_K,'--retrieval-only','--max-attempts',3,'--retry-failed','--progress']
run(retrieve,'retrieve_children')
child_records={row['question_id']:row for row in rows(CHILD_TRACE/'retrievals.jsonl') if row.get('status')=='success'}; assert set(child_records)==set(reserve_ids)
if str(PROJECT_ROOT/'common') not in sys.path: sys.path.insert(0,str(PROJECT_ROOT/'common'))
from newsqa_rag.retrieval.hierarchical import expand_ranked_children_to_parents
parent_retrievals=PERSIST/'parent_retrievals.jsonl'
with parent_retrievals.open('w',encoding='utf-8') as out:
    for qid in reserve_ids:
        record=dict(child_records[qid]); trace=dict(record['trace']); child_count=len(trace['reranked_chunks']); trace['retrieved_chunks']=expand_ranked_children_to_parents(trace['retrieved_chunks'],child_parent,parent_by_id,TOP_K); trace['reranked_chunks']=expand_ranked_children_to_parents(trace['reranked_chunks'],child_parent,parent_by_id,RERANK_TOP_N); trace['retrieved_ids']=[row['id'] for row in trace['reranked_chunks']]; trace['contexts']=[row['text'] for row in trace['reranked_chunks']]; trace['hierarchical_expansion']={'reranked_child_count':child_count,'delivered_parent_count':len(trace['reranked_chunks'])}; assert len(trace['reranked_chunks'])==RERANK_TOP_N; record['trace']=trace; out.write(json.dumps(record)+'\n')
collect=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset_parent,'--variant-manifest',parent_profile,'--config',config_path,'--run-dir',RUN_DIR,'--question-ids-file',IDS,'--chunks-path',chunks,'--bm25-path',sparse_index,'--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--generator-model',GENERATOR_MODEL,'--prompt-id',PROMPT_ID,'--system-prompt-file',prompt,'--context-depth',LOCKED_WINNER_DEPTH,'--source-retrievals',parent_retrievals,'--generation-min-interval-seconds',GENERATOR_INTERVAL,'--max-attempts',3,'--retry-failed','--progress']
run(collect,'generate',{'GEMINI_API_KEY':GENERATOR_KEY})
run([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_prejudge')
judge=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',RUN_DIR,'--judge-provider','fireworks','--judge-model',JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--question-ids-file',JUDGE_IDS,'--batch-size',1,'--max-workers',1,'--max-attempts',3,'--retry-failed','--require-complete-metrics','--progress']
run(judge,'judge',{'FIREWORKS_API_KEY':JUDGE_KEY})
run([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_final')
access.update({'status':'complete','completed_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'child_retrievals_sha256':sha(CHILD_TRACE/'retrievals.jsonl'),'parent_retrievals_sha256':sha(parent_retrievals),'predictions_sha256':sha(RUN_DIR/'predictions.jsonl'),'judge_results_sha256':sha(RUN_DIR/'judge_results.jsonl')}); access_path.write_text(json.dumps(access,indent=2,sort_keys=True)+'\n')

## 4. Reserve generalization report and export

In [ ]:
report=json.loads((RUN_DIR/'report.json').read_text()); predictions=rows(RUN_DIR/'predictions.jsonl'); judges=rows(RUN_DIR/'judge_results.jsonl'); scores=rows(RUN_DIR/'deterministic_scores.jsonl')
generation_usage={key:sum(int((row.get('result') or {}).get('usage',{}).get(key,0)) for row in predictions if row.get('status')=='success') for key in ['input_tokens','output_tokens']}
judge_batches={}
for row in judges:
    if row.get('status')=='success': judge_batches.setdefault(row.get('batch_id',row['question_id']),row.get('batch_usage') or row.get('usage') or {})
judge_usage={key:sum(int(batch.get(key,0)) for batch in judge_batches.values()) for key in ['input_tokens','output_tokens']}
cost={'generation_usd':generation_usage['input_tokens']/1e6*GENERATOR_INPUT_USD_PER_MILLION+generation_usage['output_tokens']/1e6*GENERATOR_OUTPUT_USD_PER_MILLION,'judge_usd':judge_usage['input_tokens']/1e6*JUDGE_INPUT_USD_PER_MILLION+judge_usage['output_tokens']/1e6*JUDGE_OUTPUT_USD_PER_MILLION}; cost['total_usd']=cost['generation_usd']+cost['judge_usd']
assert report['coverage']['successful']==587 and report['coverage']['failed']==0 and report.get('ragas',{}).get('n_samples')==587 and len(scores)==587
metric_paths={'qa_exact_match':('qa','exact_match'),'qa_f1':('qa','f1'),'citation_f1':('citations','citation_f1'),'citation_validity':('citations','citation_validity'),'answer_correctness':('ragas','answer_correctness'),'faithfulness':('ragas','faithfulness'),'answer_relevancy':('ragas','answer_relevancy')}
micro={name:report[group][metric] for name,(group,metric) in metric_paths.items()}
question_rows=[]
for row in scores:
    question_rows.append({'question_id':row['question_id'],'article_key':row['article_key'],'retrieval_group':'gold_in_top5' if row['retrieval']['hit_rate@5']==1 else 'gold_not_in_top5','qa_exact_match':row['qa']['exact_match'],'qa_f1':row['qa']['f1'],'citation_f1':row['citations']['citation_f1'],'citation_validity':row['citations']['citation_validity'],**{name:row['ragas'][name] for name in ('answer_correctness','faithfulness','answer_relevancy')}})
question_frame=pd.DataFrame(question_rows); metric_names=list(metric_paths); assert question_frame[metric_names].notna().all().all()
article_frame=question_frame.groupby('article_key',as_index=False)[metric_names].mean(); assert len(article_frame)==100
subgroup_frame=question_frame.groupby('retrieval_group')[metric_names].agg(['count','mean']).reset_index()
article_macro={name:round(float(article_frame[name].mean()),4) for name in metric_names}
development=finalist_records[LOCKED_WINNER_DEPTH]['summary']; development_micro={name:development[group][metric] for name,(group,metric) in metric_paths.items()}
generalization_delta={name:round(float(micro[name]-development_micro[name]),4) for name in metric_names}
summary={'schema_version':1,'partition':'heldout_reserve','selection_role':'winner_generalization_only','winner':winner_decision['winner'],'articles':100,'questions':587,'coverage':report['coverage'],'metrics_question_micro':micro,'metrics_article_macro':article_macro,'development_reference_question_micro':development_micro,'reserve_minus_development':generalization_delta,'retrieval':{'method':'bge-m3-sparse-child-to-parent','top_k':TOP_K,'reranker':'BAAI/bge-reranker-large','rerank_candidate_n':TOP_K,'parent_contexts':RERANK_TOP_N},'generator':{'model':GENERATOR_MODEL,'reasoning_effort':GENERATOR_REASONING,'max_tokens':GENERATOR_MAX_TOKENS},'judge':{'provider':'fireworks','model':JUDGE_MODEL,'reasoning_effort':JUDGE_REASONING,'max_tokens':JUDGE_MAX_TOKENS},'latency':report.get('latency',{}),'usage':{'generation':generation_usage,'judge':judge_usage},'estimated_cost':cost,'provenance':{'winner_decision_sha256':sha(decision_path),'reserve_ids_sha256':sha(IDS),'prompt_sha256':sha(prompt),'artifact_sha256':PHASE2C_SHA256,'artifact_repo':PHASE2C_REPO_ID,'artifact_revision':PHASE2C_REVISION,'repo_commit':REPO_COMMIT},'completed_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
(RESULTS/'reserve_final_summary.json').write_text(json.dumps(summary,indent=2,sort_keys=True)+'\n'); pd.json_normalize(summary).to_csv(RESULTS/'reserve_final_summary.csv',index=False)
question_frame.to_csv(RESULTS/'reserve_question_scores.csv',index=False); article_frame.to_csv(RESULTS/'reserve_article_scores.csv',index=False); subgroup_frame.to_csv(RESULTS/'reserve_retrieval_subgroups.csv',index=False)
display(pd.DataFrame([micro],index=['reserve_micro'])); display(pd.DataFrame([development_micro,micro,generalization_delta],index=['development','reserve','reserve_minus_development'])); display(subgroup_frame)
for name in ['run_manifest.json','report.json','report_summary.txt','predictions.jsonl','retrievals.jsonl','deterministic_scores.jsonl','judge_results.jsonl','attempts.jsonl','environment.json']:
    source=RUN_DIR/name
    if source.exists(): shutil.copy2(source,RESULTS/name)
bundle=Path(shutil.make_archive(str(WORK/f'phase2c_c3_p2_d{LOCKED_WINNER_DEPTH}_reserve_final_results'),'zip',root_dir=PERSIST)); destination=Path('/content/drive/MyDrive/newsqa_phase2c/results')/bundle.name; destination.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(bundle,destination)
print('Saved:',destination,'| SHA-256:',sha(destination))